# SOFR structures cross-check (Python vs BBG)

En producción, **CURVES/FLIES del live store ya vienen de BBG** (`USOS####` / `UF######`).
Este notebook compara el cálculo desde outrights vs esos tickers BBG (qa / control).

| tipo | ejemplo | ticker BBG |
|------|---------|------------|
| curve | `5s10s` | `USOS0510 Curncy` |
| fly | `5s7s10s` | `UF050710 Curncy` |

1. `git pull origin main`
2. Corre **setup → bootstrap → opciones → crosscheck**
3. Revisa el summary + columnas `*_python` / `*_bbg`
4. Excel: `sofr_structures_crosscheck.xlsx` (sheets `curves`, `flies`, `summary`)

Para refrescar el live store con estructuras BBG: `python run_dz.py prediction --once --pull-history`.


In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

logging.disable(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
for _n in list(logging.root.manager.loggerDict):
    lg = logging.getLogger(_n)
    if any(k in str(_n).lower() for k in ("bbg", "blp", "dz", "datazone", "activity", "ms")):
        lg.disabled = True
        lg.handlers = []

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 40)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "crosscheck.py").exists() else _DZ
if not (REPO / "crosscheck.py").exists():
    raise FileNotFoundError("No encuentro crosscheck.py en %s — haz git pull" % REPO)

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())
print("python:", sys.executable)

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# ── Opciones ────────────────────────────────────────────────────────────────
PULL_OUTRIGHTS = False   # False = python desde sofr_live.csv; True = también bdib outrights
START = None             # e.g. "20251001" o None = inicio del panel live
END = None               # e.g. None = ahora / fin del panel
TOL_BP = None            # None = curve_config.CROSSCHECK_TOL_BP
OUT = None               # None = sofr_structures_crosscheck.xlsx

import curve_config
import crosscheck

importlib.reload(curve_config)
importlib.reload(crosscheck)

print("curves → BBG:")
for n, t in crosscheck.curve_ticker_map().items():
    print(" ", n, "→", t)
print("flies → BBG:")
for n, t in crosscheck.fly_ticker_map().items():
    print(" ", n, "→", t)
print("tol bp:", TOL_BP or curve_config.CROSSCHECK_TOL_BP)
print("out:", OUT or curve_config.CROSSCHECK_PATH)

In [ ]:
# ── Corre el cross-check (pide BBG USOS / UF por bdib) ───────────────────────
curves_df, flies_df, summary, path = crosscheck.run_crosscheck(
    from_store=not PULL_OUTRIGHTS,
    start_date=START,
    end_date=END,
    path=OUT,
    tol=TOL_BP,
)
print("Excel:", path)

In [ ]:
# ── Summary: qué estructuras coinciden ──────────────────────────────────────
display(summary)

In [ ]:
# ── Curves: python | bbg lado a lado (últimas filas) ─────────────────────────
print("curves shape:", curves_df.shape)
display(curves_df.tail(15))

In [ ]:
# ── Flies: python | bbg lado a lado (últimas filas) ──────────────────────────
print("flies shape:", flies_df.shape)
display(flies_df.tail(15))

In [ ]:
# ── Diff rápido de una estructura (cambia NAME) ─────────────────────────────
NAME = "5s7s10s"  # e.g. "5s10s", "2s5s10s", "5s7s10s"

py_c, bb_c = "%s_python" % NAME, "%s_bbg" % NAME
src = flies_df if py_c in flies_df.columns else curves_df
if py_c not in src.columns:
    raise KeyError("%s not in curves/flies — elige un NAME del summary" % NAME)

chk = src[[py_c, bb_c]].dropna().copy()
chk["diff"] = chk[py_c] - chk[bb_c]
chk["abs_diff"] = chk["diff"].abs()
print(NAME, "n=", len(chk), "max|diff|=", float(chk["abs_diff"].max()))
display(chk.tail(20))
display(chk.nlargest(5, "abs_diff"))